# Ukrainian Folk Art Annotations: JSON to CSV (combined)

This notebook, similarly to 3.01, reads all json files from /data and saves annotations from all 4 events to a combined csv file.

## Setup
Define a common `BASE_DIR` for file paths and, when running in Google Colab, optionally mount Google Drive.

In [8]:
from pathlib import Path



IN_COLAB = False
try:
  from google.colab import drive  # type: ignore
  IN_COLAB = True
except ImportError:
  pass

if IN_COLAB:
  # Optional in Colab if you need a newer package version:
  # !pip -q install pandas

  drive.mount('/content/drive')

USE_DRIVE = IN_COLAB  # set False to use local runtime files

if USE_DRIVE:
  BASE_DIR = Path('/content/drive/MyDrive/AISTER-Crowdsourcing-Pilot/step_3')
else:
  BASE_DIR = Path('../')

In [9]:
print(f'Base directory: {BASE_DIR.resolve()}')

Base directory: C:\Users\User\Desktop\Web2Learn\AISTER-Crowdsourcing-Pilot\step_3


## Setup

Load json data from all 4 files into separate lists.

In [10]:
import json

data_dir = BASE_DIR / 'data'
json_files = sorted(data_dir.glob('*.json'))

assert len(json_files) == 4, f'Expected 4 json files in {data_dir}, found {len(json_files)}'

data1, data2, data3, data4 = [json.load(f.open('r', encoding='utf-8')) for f in json_files]

graph1, graph2, graph3, graph4 = data1.get('@graph') or [], data2.get('@graph') or [], data3.get('@graph') or [], data4.get('@graph') or []

print(f'Loaded data from {len(json_files)} files:')
for i, f in enumerate(json_files, 1):
    print(f'  {i}. {f.name} with {len(graph1) if i == 1 else len(graph2) if i == 2 else len(graph3) if i == 3 else len(graph4)} graph items')


Loaded data from 4 files:
  1. 1_ukrainian-folkart_annotations.json with 4018 graph items
  2. 2_ukrainian-folkart_annotations.json with 4018 graph items
  3. 3_ukrainian-folkart_annotations.json with 4760 graph items
  4. 4_ukrainian-folkart_annotations.json with 5829 graph items


## Transform annotations

Build 4 separate normalized data frames with default values where needed

In [ ]:
import pandas as pd


def build_annotations_df(graph_items):
  rows = []

  for g in graph_items:
    source = (g.get('target') or {}).get('source') or ''

    rows.append({
      'created': g.get('created') or '',
      'value': (g.get('body') or {}).get('value'),
      'europeana_id': source.split('/')[-1],
      'upvotes': (g.get('review') or {}).get('upvotes') or 0,
      'downvotes': (g.get('review') or {}).get('downvotes') or 0,
      'recommendation': (g.get('review') or {}).get('recommendation') or 'unknown',
    })

  df_local = pd.DataFrame(rows)

  created_dt = pd.to_datetime(
    df_local['created'],
    format='%a %b %d %H:%M:%S UTC %Y',
    errors='coerce',
    utc=True,
  )

  df_local['created'] = created_dt.dt.strftime('%d.%m.%Y')
  df_local['upvotes'] = df_local['upvotes'].astype('int')
  df_local['downvotes'] = df_local['downvotes'].astype('int')

  df_local = df_local.sort_values(by='europeana_id').reset_index(drop=True)

  return df_local

df1 = build_annotations_df(graph1)
df2 = build_annotations_df(graph2)
df3 = build_annotations_df(graph3)
df4 = build_annotations_df(graph4)

display(df1.head())
display(df2.head())
display(df3.head())
display(df4.head())

,created,value,europeana_id,upvotes,downvotes,recommendation
0,13.02.2026,geometric patterns,HMT1,1,0,accept
1,13.02.2026,icon,HMT1,1,0,accept
2,13.02.2026,wedding couple,HMT1,1,0,accept
3,13.02.2026,traditional iconographic style,HMT1,0,0,unknown
4,13.02.2026,religious icon,HMT1,2,0,accept


,created,value,europeana_id,upvotes,downvotes,recommendation
0,13.02.2026,heart,HMT1,1,0,accept
1,13.02.2026,traditional iconographic style,HMT1,0,0,unknown
2,13.02.2026,couple,HMT1,1,0,accept
3,13.02.2026,decorative wooden border,HMT1,1,0,accept
4,13.02.2026,cross,HMT1,1,0,accept


,created,value,europeana_id,upvotes,downvotes,recommendation
0,27.03.2026,Blue veil,HMT1,2,0,accept
1,13.02.2026,wedding couple,HMT1,4,2,accept
2,13.02.2026,halos,HMT1,4,0,accept
3,13.02.2026,geometric patterns,HMT1,5,0,accept
4,13.02.2026,mary,HMT1,4,0,accept


,created,value,europeana_id,upvotes,downvotes,recommendation
0,13.02.2026,mary,HMT1,10,1,accept
1,13.02.2026,jesus,HMT1,10,0,accept
2,13.02.2026,heart,HMT1,11,0,accept
3,13.02.2026,decorative wooden border,HMT1,11,0,accept
4,13.02.2026,halos,HMT1,11,0,accept


## Combine data frames

Combine all 4 dataframes in one, keeping track of the event number in which the annotation was last edited (most recent one is event no. 4)

In [12]:
combined_df = pd.concat(
    [
        df1.assign(event_number=1),
        df2.assign(event_number=2),
        df3.assign(event_number=3),
        df4.assign(event_number=4),
    ],
    ignore_index=True,
)

combined_df = (
    combined_df
    .sort_values(['created', 'value', 'event_number'], ascending=[True, True, False])
    .drop_duplicates(subset=['created', 'value'], keep='first')
    .reset_index(drop=True)
)

display(combined_df.head())
print(f"Total unique annotations: {len(combined_df)}")


,created,value,europeana_id,upvotes,downvotes,recommendation,event_number
0,01.04.2026,8 episodes,KYD1802,1,0,accept,4
1,01.04.2026,Birch trees,KYD1831,4,0,accept,4
2,01.04.2026,Black background,KYD1790,5,1,accept,4
3,01.04.2026,Blurred dark background,KYD1846,2,0,accept,4
4,01.04.2026,Dark background,KYD1822,4,1,accept,4


Total unique annotations: 2475


## Save to csv

Save combined data frame to csv

In [13]:
combined_df.to_csv('../outputs/ukrainian-folkart-annotations.csv', index=False)